# 邮件营销实验分析：发什么邮件、发给谁、值不值得发？

**背景：** 一家服装零售商对过去 12 个月有过购买的 64,000 名客户做了一次随机实验：三分之一收到**男装促销邮件**，三分之一收到**女装促销邮件**，三分之一**不发邮件**（对照组），之后两周追踪客户是否访问网站、是否下单、消费多少。

**要回答的业务问题：**
1. 两种邮件分别带来了多少增量？哪种更好？
2. 能否用实验前的客户数据，让结论更精确？（方差缩减）
3. 邮件对哪类客户最有效？（异质性分析）
4. 最终应该给谁发、发哪种？考虑成本后是否赚钱？（增量建模与投放决策）

**指标：**

| 类型 | 指标 | 说明 |
|---|---|---|
| 核心指标 | 人均消费 `spend` | 两周内每位客户的消费金额（美元），直接对应收入 |
| 次要指标 | 访问率 `visit`、转化率 `conversion` | 是否访问网站、是否下单 |

**实验前特征（不受实验影响，可以用来分层和调整）：** 距上次购买月数 `recency`、过去一年消费金额 `history`、是否买过男装 `mens` / 女装 `womens`、是否新客 `newbie`、地区类型 `zip_code`、购买渠道 `channel`

**数据来源：** Kevin Hillstrom, MineThatData E-Mail Analytics and Data Mining Challenge (2008)。

In [ ]:
# 在 Google Colab 上运行时先执行这一格；在自己电脑上运行会自动跳过
import os, sys, subprocess, zipfile
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    root = "/content/drive/MyDrive/email-marketing-experiment"
    if not os.path.exists(root):
        zipfile.ZipFile("/content/drive/MyDrive/email-marketing-experiment.zip").extractall("/content/drive/MyDrive")
    os.chdir(root + "/notebooks")
    subprocess.run("pip -q install statsmodels", shell=True, check=True)
    subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null", shell=True, check=True)
    import matplotlib.font_manager as fm
    for f in fm.findSystemFonts():
        if "NotoSansCJK" in f:
            fm.fontManager.addfont(f)
    print("Colab 环境准备完成，当前目录：", os.getcwd())

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

ROOT = Path("..").resolve()
DATA = ROOT / "data" / "hillstrom.csv"
FIG_DIR, RES_DIR = ROOT / "figures", ROOT / "results"
FIG_DIR.mkdir(exist_ok=True); RES_DIR.mkdir(exist_ok=True)

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti TC", "Arial Unicode MS", "Microsoft YaHei",
                                   "SimHei", "Noto Sans CJK SC", "Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

GROUPS = ["control", "mens", "womens"]
LABEL = {"control": "不发邮件", "mens": "男装邮件", "womens": "女装邮件"}
COLORS = {"control": "#9AA5B1", "mens": "#3B6FB6", "womens": "#E07B39"}
ALPHA = 0.05
SEED = 42
rng = np.random.default_rng(SEED)
summary = {}


def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)


df = pd.read_csv(DATA)
df["group"] = df["segment"].map({"No E-Mail": "control", "Mens E-Mail": "mens", "Womens E-Mail": "womens"})
df["buyer_type"] = np.select([(df["mens"] == 1) & (df["womens"] == 1), df["mens"] == 1, df["womens"] == 1],
                             ["男女装都买过", "只买过男装", "只买过女装"], "都没买过")
print(f"客户数：{len(df):,}")
df.head()

## 1. 数据检查

In [ ]:
buyers = df[df["spend"] > 0]
check = pd.Series({
    "客户数": len(df),
    "缺失值": int(df.isna().sum().sum()),
    "整体访问率": f"{df['visit'].mean():.2%}",
    "整体转化率": f"{df['conversion'].mean():.2%}",
    "有消费的客户占比": f"{(df['spend'] > 0).mean():.2%}",
    "有消费客户的平均消费": f"{buyers['spend'].mean():.2f} 美元",
    "有消费客户的消费中位数": f"{buyers['spend'].median():.2f} 美元",
    "单笔最高消费": f"{df['spend'].max():.2f} 美元",
})
display(check.to_frame("值"))
print("客户类型分布：")
df["buyer_type"].value_counts().to_frame("客户数")

**数据特点：** 只有不到 1% 的客户下单，人均消费 99% 以上是 0，有消费的客户金额从 30 美元到 499 美元不等。核心指标噪声很大，所以检验均值时要用对分布假设更宽松的方法（韦尔奇 t 检验 + 自助法）。

## 2. 实验质量检查

### 2.1 分流检查

三组设计为各占三分之一，用卡方拟合优度检验确认实际人数没有异常偏离。

In [ ]:
counts = df["group"].value_counts().reindex(GROUPS)
chi2_srm, p_srm = stats.chisquare(counts)
display(pd.DataFrame({"人数": counts, "占比": counts / len(df)}).rename(index=LABEL))
print(f"卡方值 = {chi2_srm:.3f}，P 值 = {p_srm:.3f}："
      + ("分流比例正常。" if p_srm >= 0.01 else "⚠️ 存在样本比例不匹配，需要先排查。"))
summary["分流检查"] = {LABEL[g]: int(counts[g]) for g in GROUPS} | {"P值": round(float(p_srm), 3)}

### 2.2 协变量平衡检查

随机分组成功的话，三组客户在**实验前**的特征上应该没有系统差异。用**标准化均值差**衡量：绝对值小于 0.1 视为平衡良好。

In [ ]:
feat = pd.get_dummies(df[["recency", "history", "mens", "womens", "newbie", "zip_code", "channel"]],
                      columns=["zip_code", "channel"], dtype=float)
ctrl = feat[df["group"] == "control"]
smd = {}
for g in ["mens", "womens"]:
    t = feat[df["group"] == g]
    smd[f"{LABEL[g]} vs 不发邮件"] = (t.mean() - ctrl.mean()) / np.sqrt((t.var() + ctrl.var()) / 2)
smd = pd.DataFrame(smd)
max_smd = smd.abs().max().max()
print(f"所有特征的标准化均值差绝对值最大为 {max_smd:.3f}，" + ("全部小于 0.1，三组平衡良好。" if max_smd < 0.1 else "存在不平衡的特征。"))
summary["协变量平衡_最大标准化均值差"] = round(float(max_smd), 4)
smd

## 3. 主效应：两种邮件各带来多少增量？

- 访问率、转化率：两样本比例 Z 检验
- 人均消费：韦尔奇 t 检验（不假设两组方差相等），并用自助法交叉验证
- 共 3 个指标 × 3 组对比 = 9 次检验，用 **Holm 方法**做多重检验校正

In [ ]:
def prop_test(y_t, y_c):
    p1, p0, n1, n0 = y_t.mean(), y_c.mean(), len(y_t), len(y_c)
    pp = (y_t.sum() + y_c.sum()) / (n1 + n0)
    z = (p1 - p0) / np.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n0))
    se = np.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)
    return p1 - p0, se, 2 * (1 - stats.norm.cdf(abs(z)))


def welch_test(y_t, y_c):
    t, p = stats.ttest_ind(y_t, y_c, equal_var=False)
    se = np.sqrt(y_t.var(ddof=1) / len(y_t) + y_c.var(ddof=1) / len(y_c))
    return y_t.mean() - y_c.mean(), se, p


def boot_diff(y_t, y_c, B=2000):
    y_t, y_c = np.asarray(y_t), np.asarray(y_c)
    d = np.array([y_t[rng.integers(0, len(y_t), len(y_t))].mean() - y_c[rng.integers(0, len(y_c), len(y_c))].mean()
                  for _ in range(B)])
    return np.percentile(d, [2.5, 97.5])


comparisons = [("mens", "control"), ("womens", "control"), ("mens", "womens")]
rows = []
for t, c in comparisons:
    for metric in ["spend", "visit", "conversion"]:
        y_t, y_c = df.loc[df["group"] == t, metric], df.loc[df["group"] == c, metric]
        diff, se, p = (welch_test if metric == "spend" else prop_test)(y_t, y_c)
        row = {"对比": f"{LABEL[t]} vs {LABEL[c]}", "指标": metric, "实验组": y_t.mean(), "对照组": y_c.mean(),
               "差值": diff, "95%置信区间下限": diff - 1.96 * se, "95%置信区间上限": diff + 1.96 * se,
               "相对提升": diff / y_c.mean(), "P值": p}
        if metric == "spend":
            row["自助法区间下限"], row["自助法区间上限"] = boot_diff(y_t, y_c)
        rows.append(row)
effects = pd.DataFrame(rows)

# Holm 校正
order = effects["P值"].sort_values().index
m = len(order)
adj = pd.Series([min(1, effects.loc[i, "P值"] * (m - k)) for k, i in enumerate(order)], index=order).cummax()
effects["Holm校正后P值"] = adj
effects["显著"] = effects["Holm校正后P值"] < ALPHA
effects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
titles = {"spend": "人均消费（美元）", "visit": "访问率（%）", "conversion": "转化率（%）"}
for ax, metric in zip(axes, ["spend", "visit", "conversion"]):
    g = df.groupby("group")[metric]
    mean, se = g.mean().reindex(GROUPS), (g.std() / np.sqrt(g.size())).reindex(GROUPS)
    scale = 1 if metric == "spend" else 100
    ax.bar([LABEL[x] for x in GROUPS], mean * scale, yerr=1.96 * se * scale, capsize=6,
           color=[COLORS[x] for x in GROUPS], width=0.6)
    for i, v in enumerate(mean * scale):
        ax.text(i, v * 0.5, f"{v:.2f}", ha="center", color="white", fontweight="bold")
    ax.set_title(titles[metric])
fig.suptitle("三组的核心指标对比（误差线为 95% 置信区间）", y=1.03)
fig.tight_layout()
save_fig(fig, "01_main_effects")

## 4. 方差缩减：用实验前数据让结论更精确

**思路：** 实验组和对照组的差异估计有噪声，噪声里有一部分来自客户本身的差异（比如老客户本来就爱买）。如果用实验前就知道的特征把这部分差异"扣掉"，估计会更精确，同样的样本量能检测出更小的效果。

两种做法：
- **CUPED**（业界实验平台常用）：用一个实验前变量 X 调整结果，Y′ = Y − θ(X − X̄)，θ = Cov(Y, X) / Var(X)。理论上方差缩减比例约等于 Y 和 X 的相关系数平方
- **回归调整**：用全部实验前特征做回归 Y ~ 实验分组 + 特征，取分组系数，使用稳健标准误（HC2）

In [ ]:
df["log_history"] = np.log1p(df["history"])
covars = "recency + log_history + mens + womens + newbie + C(zip_code) + C(channel)"
vr_rows = []
for arm in ["mens", "womens"]:
    d = df[df["group"].isin(["control", arm])].copy()
    d["T"] = (d["group"] == arm).astype(int)
    for metric in ["spend", "visit"]:
        naive = smf.ols(f"{metric} ~ T", d).fit(cov_type="HC2")
        # CUPED：以实验前消费金额（取对数）为协变量
        x = d["log_history"]
        theta = np.cov(d[metric], x)[0, 1] / x.var()
        d["y_cuped"] = d[metric] - theta * (x - x.mean())
        cuped = smf.ols("y_cuped ~ T", d).fit(cov_type="HC2")
        reg = smf.ols(f"{metric} ~ T + {covars}", d).fit(cov_type="HC2")
        rho = np.corrcoef(d[metric], x)[0, 1]
        vr_rows.append({
            "对比": f"{LABEL[arm]} vs 不发邮件", "指标": metric,
            "原始估计": naive.params["T"], "原始标准误": naive.bse["T"],
            "CUPED标准误": cuped.bse["T"], "回归调整估计": reg.params["T"], "回归调整标准误": reg.bse["T"],
            "相关系数平方": rho ** 2,
            "CUPED方差缩减": 1 - (cuped.bse["T"] / naive.bse["T"]) ** 2,
            "回归调整方差缩减": 1 - (reg.bse["T"] / naive.bse["T"]) ** 2,
        })
vr = pd.DataFrame(vr_rows)
vr

**解读：** 方差缩减效果很小（访问率约 3%，消费几乎为 0）。原因正如公式所示：方差缩减比例约等于结果和协变量相关系数的平方，而**过去一年的消费金额和这两周的行为相关性很弱**（相关系数约 0.07，平方约 0.5%）；加上其他特征后也只到 3% 左右。

CUPED 真正好用的场景，是有**同一指标的实验前数据**，比如用"实验前两周的访问次数"去调整"实验期两周的访问次数"，相关系数往往有 0.5 到 0.7，能缩减 25% 到 50% 的方差。这个数据集没有这类变量，所以这里的结论是：**方差缩减方法本身没问题，但效果取决于协变量的预测力**，这也是实际工作中选择协变量的依据。

## 5. 异质性分析：邮件对哪类客户最有效？

按**实验前**的特征分组，分别估计两种邮件的效果。这些特征在发邮件之前就确定了，不受实验影响，所以分组比较是合法的（反过来，按实验后的行为分组会引入偏差）。

> 子组很多时会产生多重比较问题，这里的结果属于**探索性分析**，用于提出假设，正式上线前需要再验证。

In [ ]:
def lift_ci(d, arm, metric):
    y_t, y_c = d.loc[d["group"] == arm, metric], d.loc[d["group"] == "control", metric]
    diff, se, p = (welch_test if metric == "spend" else prop_test)(y_t, y_c)
    return diff, diff - 1.96 * se, diff + 1.96 * se, len(y_t) + len(y_c)


seg_rows = []
for col, name in [("buyer_type", "客户类型"), ("newbie", "是否新客"), ("channel", "购买渠道"), ("zip_code", "地区类型")]:
    for val, d in df.groupby(col):
        for arm in ["mens", "womens"]:
            for metric in ["visit", "spend"]:
                diff, lo, hi, n = lift_ci(d, arm, metric)
                seg_rows.append({"维度": name, "分组": str(val), "邮件": LABEL[arm], "指标": metric,
                                 "提升": diff, "下限": lo, "上限": hi, "样本": n})
seg = pd.DataFrame(seg_rows)
seg_visit = seg[seg["指标"] == "visit"].pivot_table(index=["维度", "分组"], columns="邮件", values="提升") * 100
print("访问率提升（百分点）：")
seg_visit.round(2)

In [ ]:
# 森林图：按客户类型看两种邮件对访问率的提升
s = seg[(seg["维度"] == "客户类型") & (seg["指标"] == "visit")]
fig, ax = plt.subplots(figsize=(8, 3.6))
types = list(s["分组"].unique())
for j, (arm, off) in enumerate([("男装邮件", -0.12), ("女装邮件", 0.12)]):
    r = s[s["邮件"] == arm].set_index("分组").reindex(types)
    y = np.arange(len(types)) + off
    ax.errorbar(r["提升"] * 100, y, xerr=[(r["提升"] - r["下限"]) * 100, (r["上限"] - r["提升"]) * 100],
                fmt="o", capsize=4, color=COLORS["mens" if j == 0 else "womens"], label=arm)
ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_yticks(range(len(types)))
ax.set_yticklabels([f"{t}（{(df['buyer_type'] == t).sum():,} 人）" for t in types])
ax.set_xlabel("访问率提升（百分点，95% 置信区间）")
ax.set_title("邮件效果的异质性：女装邮件只对买过女装的客户有效")
ax.legend()
save_fig(fig, "02_heterogeneity_buyer_type")

In [ ]:
# 用交互项回归正式检验：女装邮件的效果是否取决于客户是否买过女装
d = df[df["group"].isin(["control", "womens"])].copy()
d["T"] = (d["group"] == "womens").astype(int)
inter = smf.ols("visit ~ T * womens", d).fit(cov_type="HC2")
print(f"女装邮件 × 买过女装 的交互项：{inter.params['T:womens'] * 100:.2f} 个百分点，P = {inter.pvalues['T:womens']:.2g}")
d = df[df["group"].isin(["control", "mens"])].copy()
d["T"] = (d["group"] == "mens").astype(int)
inter_m = smf.ols("visit ~ T * mens", d).fit(cov_type="HC2")
print(f"男装邮件 × 买过男装 的交互项：{inter_m.params['T:mens'] * 100:.2f} 个百分点，P = {inter_m.pvalues['T:mens']:.2g}")
summary["异质性_交互项"] = {
    "女装邮件x买过女装_百分点": round(float(inter.params["T:womens"] * 100), 2), "女装P值": float(inter.pvalues["T:womens"]),
    "男装邮件x买过男装_百分点": round(float(inter_m.params["T:mens"] * 100), 2), "男装P值": float(inter_m.pvalues["T:mens"])}
summary["异质性_客户类型_访问率提升_百分点"] = {
    f"{r['分组']}_{r['邮件']}": round(float(r["提升"] * 100), 2) for _, r in s.iterrows()}

## 6. 增量建模：能不能预测"谁会被邮件打动"？

**方法：** T-Learner。把数据按 50:50 分成训练集和测试集；在训练集上，对三组分别训练一个梯度提升模型，预测客户在该组的访问概率。对测试集的每位客户，"发男装邮件的预测访问率 − 不发邮件的预测访问率"就是预测的增量。

**检验方法：** 把测试集客户按预测增量从低到高分成 5 组，看每组**真实的**增量（实验组均值 − 对照组均值）是否也从低到高。如果是，说明模型能识别出"容易被打动的人"。

> 这里用访问率而不是消费金额建模：只有不到 1% 的客户有消费，用消费金额建模噪声太大，模型几乎学不到东西。

In [ ]:
X = pd.get_dummies(df[["recency", "history", "mens", "womens", "newbie", "zip_code", "channel"]],
                   columns=["zip_code", "channel"], dtype=float)
train_idx, test_idx = train_test_split(df.index, test_size=0.5, stratify=df["group"], random_state=SEED)
pred = {}
for g in GROUPS:
    idx = [i for i in train_idx if df.at[i, "group"] == g]
    model = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=200,
                                          min_samples_leaf=300, random_state=SEED)
    model.fit(X.loc[idx], df.loc[idx, "visit"])
    pred[g] = model.predict(X.loc[test_idx])
pred = pd.DataFrame(pred, index=test_idx)
test = df.loc[test_idx].copy()
test["uplift_mens"] = pred["mens"] - pred["control"]
test["uplift_womens"] = pred["womens"] - pred["control"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
quint = {}
for ax, arm in zip(axes, ["mens", "womens"]):
    d = test[test["group"].isin(["control", arm])].copy()
    d["五分组"] = pd.qcut(d[f"uplift_{arm}"].rank(method="first"), 5, labels=[f"第{i}组" for i in range(1, 6)])
    actual = d.groupby("五分组", observed=True).apply(
        lambda g: g.loc[g["group"] == arm, "visit"].mean() - g.loc[g["group"] == "control", "visit"].mean()) * 100
    quint[arm] = actual
    ax.bar(actual.index.astype(str), actual.values, color=COLORS[arm])
    for i, v in enumerate(actual.values):
        ax.text(i, v + 0.2, f"{v:.1f}", ha="center")
    ax.set(title=f"{LABEL[arm]}：按预测增量分 5 组的真实增量", xlabel="预测增量从低到高",
           ylabel="真实访问率提升（百分点）")
fig.tight_layout()
save_fig(fig, "03_uplift_quintiles")
summary["增量模型_测试集五分组真实提升_百分点"] = {LABEL[a]: [round(float(v), 2) for v in q.values] for a, q in quint.items()}
pd.DataFrame(quint).rename(columns=LABEL).round(2)

**解读：** 女装邮件的真实增量随预测增量明显上升，说明模型抓住了"买过女装的人更容易被女装邮件打动"这一规律；男装邮件的真实增量在各组之间没有单调趋势，因为第 5 节已经看到，男装邮件对各类客户的效果都差不多。**增量模型只在效果确实存在异质性时才有价值**，建模之前先做异质性分析，可以避免白费功夫。

## 7. 投放决策：给谁发、发哪种、赚不赚钱？

**利润口径（假设，可按实际业务修改）：**
- 毛利率 `MARGIN` = 30%，即每 1 美元销售额带来 0.3 美元毛利
- 每封邮件成本 `EMAIL_COST` = 0.10 美元（制作、发送和退订损失的折算）

单个客户的利润 = 毛利率 × 消费金额 − 邮件成本（不发邮件则为 0）。

**策略评估方法：** 在测试集上用**逆概率加权**评估每个策略：因为每位客户被随机分到三组的概率都是 1/3，只要客户实际所在的组正好是策略会选的组，就用 3 倍权重计入。这样不需要真的再做一次实验，就能无偏地估计"如果按这个策略发邮件，每位客户的平均利润是多少"。再用自助法给出置信区间。

**比较的策略：**
1. 全部不发
2. 全部发男装邮件
3. 全部发女装邮件
4. 规则策略：只买过女装的客户发女装邮件，其余发男装邮件（来自第 5 节的发现）
5. 模型策略：每位客户发预测访问增量更高的那种邮件

In [ ]:
MARGIN, EMAIL_COST = 0.30, 0.10

profit = MARGIN * test["spend"] - EMAIL_COST * (test["group"] != "control")
policies = {
    "全部不发": pd.Series("control", index=test.index),
    "全部发男装邮件": pd.Series("mens", index=test.index),
    "全部发女装邮件": pd.Series("womens", index=test.index),
    "规则策略": pd.Series(np.where(test["buyer_type"] == "只买过女装", "womens", "mens"), index=test.index),
    "模型策略": pd.Series(np.where(test["uplift_womens"] > test["uplift_mens"], "womens", "mens"), index=test.index),
}
B = 1000
boot_idx = [rng.integers(0, len(test), len(test)) for _ in range(B)]
base_vals = (profit * (test["group"] == "control") * 3).to_numpy()
mens_vals = (profit * (test["group"] == "mens") * 3).to_numpy()
pol_rows = []
for name, pol in policies.items():
    match = (test["group"] == pol).to_numpy()
    v_profit = profit.to_numpy() * match * 3
    v_visit = test["visit"].to_numpy() * match * 3
    inc = np.array([(v_profit[i] - base_vals[i]).mean() for i in boot_idx]) * 1000
    vs_mens = np.array([(v_profit[i] - mens_vals[i]).mean() for i in boot_idx]) * 1000
    pol_rows.append({"策略": name,
                     "每千名客户利润（美元）": v_profit.mean() * 1000,
                     "每千名客户访问数": v_visit.mean() * 1000,
                     "比不发多赚（美元/千人）": inc.mean(), "多赚_95%下限": np.percentile(inc, 2.5),
                     "多赚_95%上限": np.percentile(inc, 97.5),
                     "比全发男装_95%下限": np.percentile(vs_mens, 2.5), "比全发男装_95%上限": np.percentile(vs_mens, 97.5),
                     "发女装邮件比例": (pol == "womens").mean()})
pol_df = pd.DataFrame(pol_rows).set_index("策略")
pol_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
r = pol_df.drop(index="全部不发")
colors = ["#3B6FB6", "#E07B39", "#4C9A6A", "#8E6BB8"]
ax.barh(r.index[::-1], r["比不发多赚（美元/千人）"][::-1], color=colors[::-1],
        xerr=[(r["比不发多赚（美元/千人）"] - r["多赚_95%下限"])[::-1], (r["多赚_95%上限"] - r["比不发多赚（美元/千人）"])[::-1]],
        capsize=5)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("与全部不发相比，每千名客户多赚的毛利（美元，95% 自助法区间）")
ax.set_title(f"投放策略评估（测试集，毛利率 {MARGIN:.0%}，每封邮件 {EMAIL_COST:.2f} 美元）")
save_fig(fig, "04_policy_value")

In [ ]:
summary["主效应"] = {
    f"{r['对比']}_{r['指标']}": {"差值": round(float(r["差值"]), 5), "相对提升": round(float(r["相对提升"]), 4),
                               "P值": float(r["P值"]), "Holm校正后P值": float(r["Holm校正后P值"])}
    for _, r in effects.iterrows()}
summary["方差缩减"] = {f"{r['对比']}_{r['指标']}": {"CUPED": round(float(r["CUPED方差缩减"]), 4),
                                                  "回归调整": round(float(r["回归调整方差缩减"]), 4),
                                                  "相关系数平方": round(float(r["相关系数平方"]), 4)}
                    for _, r in vr.iterrows()}
summary["投放策略_每千人"] = {k: {"利润": round(float(v["每千名客户利润（美元）"]), 1),
                              "比不发多赚": round(float(v["比不发多赚（美元/千人）"]), 1),
                              "多赚区间": [round(float(v["多赚_95%下限"]), 1), round(float(v["多赚_95%上限"]), 1)],
                              "比全发男装区间": [round(float(v["比全发男装_95%下限"]), 1), round(float(v["比全发男装_95%上限"]), 1)]}
                          for k, v in pol_df.iterrows()}
summary["利润假设"] = {"毛利率": MARGIN, "每封邮件成本": EMAIL_COST}
(RES_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写入 results/summary.json")

## 8. 结论与建议（根据上面的结果填写）

**主效应：**

**方差缩减：**

**异质性：**

**增量模型与投放决策：**

**建议：**

**局限：**
- 毛利率和邮件成本是假设值，结论对这两个参数敏感，上线前要用真实财务数据重算
- 只有两周的效果，看不到邮件对长期复购和退订的影响
- 异质性分析是探索性的，规则策略来自同一份数据的发现，正式采用前应再做一次实验验证